In [74]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("satyanshgaur1/rain-prediction-training-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\vivek\.cache\kagglehub\datasets\satyanshgaur1\rain-prediction-training-dataset\versions\1


In [75]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, roc_auc_score, 
    confusion_matrix, precision_score, classification_report
)
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [76]:
# HYPERPARAMTER TUNING:
RandomState = 42

In [77]:
data = pd.read_parquet(r"C:\Learning_ML\ML_Projects\Raining_Rate\test.parquet")
data.head(25)

,timestamp,received_snr_db,carrier_frequency_ghz,elevation_angle_deg,slant_range_km,fspl_db,gaseous_attenuation_db,excess_attenuation_db,effective_path_length_km,specific_attenuation_db_per_km,...,snr_roll_mean_5min,snr_roll_std_5min,snr_roll_max_5min,snr_roll_min_5min,snr_roll_mean_30min,snr_roll_std_30min,attenuation_roll_mean,attenuation_roll_std,attenuation_delta,snr_delta
0,2026-07-09 20:30:00+00:00,14.404574,10.0,-29.300419,45281.706926,205.568456,2.249753,0.823659,41.541219,0.019828,...,14.121644,0.501257,14.499212,13.270402,13.751510,1.412166,1.107207,0.501583,-0.047118,0.046808
1,2026-07-09 20:31:00+00:00,11.477863,10.0,-29.317290,45283.330661,205.568767,2.249753,3.750058,41.541219,0.090273,...,13.763136,1.287183,14.499212,11.477863,13.789541,1.330009,1.465406,1.286839,2.926399,-2.926710
2,2026-07-09 20:32:00+00:00,12.445977,10.0,-29.334242,45284.962169,205.569080,2.249753,2.781632,41.541219,0.066961,...,13.352489,1.320729,14.404574,11.477863,13.945135,0.748352,1.875743,1.320368,-0.968426,0.968113
3,2026-07-09 20:33:00+00:00,10.889104,10.0,-29.351274,45286.601388,205.569395,2.249753,4.338190,41.541219,0.104431,...,12.715057,1.619422,14.404574,10.889104,13.869826,0.924294,2.512864,1.618994,1.556558,-1.556872
4,2026-07-09 20:34:00+00:00,9.179920,10.0,-29.368385,45288.248297,205.569710,2.249753,6.047059,41.541219,0.145568,...,11.679488,1.931761,14.404574,9.179920,13.712355,1.259795,3.548120,1.931313,1.708869,-1.709185
5,2026-07-09 20:35:00+00:00,13.184948,10.0,-29.385576,45289.902862,205.570028,2.249753,2.041713,41.541219,0.049149,...,11.435562,1.538637,13.184948,9.179920,13.655421,1.243083,3.791731,1.538645,-4.005346,4.005028
6,2026-07-09 20:36:00+00:00,14.735852,10.0,-29.402845,45291.565045,205.570347,2.249753,0.490490,41.541219,0.011807,...,12.087160,2.135231,14.735852,9.179920,13.664505,1.250167,3.139817,2.135487,-1.551223,1.550904
7,2026-07-09 20:37:00+00:00,14.095035,10.0,-29.420194,45293.234813,205.570667,2.249753,1.130988,41.541219,0.027226,...,12.416972,2.323563,14.735852,9.179920,13.652941,1.244318,2.809688,2.323972,0.640498,-0.640818
8,2026-07-09 20:38:00+00:00,14.665222,10.0,-29.437620,45294.912128,205.570988,2.249753,0.560479,41.541219,0.013492,...,13.172195,2.316474,14.735852,9.179920,13.675142,1.256590,2.054146,2.316883,-0.570509,0.570187
9,2026-07-09 20:39:00+00:00,14.973057,10.0,-29.455125,45296.596954,205.571312,2.249753,0.252321,41.541219,0.006074,...,14.330823,0.717106,14.973057,13.184948,13.681620,1.262960,0.895198,0.717498,-0.308158,0.307835


In [78]:
data.shape

(90000, 36)

In [79]:
data.describe()

,received_snr_db,carrier_frequency_ghz,elevation_angle_deg,slant_range_km,fspl_db,gaseous_attenuation_db,excess_attenuation_db,effective_path_length_km,specific_attenuation_db_per_km,rain_height_km,...,snr_roll_mean_5min,snr_roll_std_5min,snr_roll_max_5min,snr_roll_min_5min,snr_roll_mean_30min,snr_roll_std_30min,attenuation_roll_mean,attenuation_roll_std,attenuation_delta,snr_delta
count,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,...,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000
mean,-0.399318,17.200000,-34.751821,45364.904534,209.578290,7.568407,7.172921,25.516169,0.344124,4.236875,...,-0.396940,3.296941,2.954988,-4.971158,-0.360110,5.943030,7.169918,3.296501,0.001827,-0.001514
std,21.148280,7.222228,28.867227,3001.909580,3.451828,7.445731,16.239047,10.202880,0.895167,0.604960,...,19.966699,7.050733,15.111366,27.892956,18.096469,9.195723,14.668422,7.050936,8.432245,8.432246
min,-412.186770,10.000000,-62.352039,39643.295735,204.413395,0.426283,-0.107632,8.799202,0.000000,3.350000,...,-283.647479,0.000392,-160.590569,-412.186770,-123.696437,0.001937,-0.040906,0.000304,-239.286621,-219.736540
25%,-8.896456,12.000000,-57.625798,44212.990446,206.936748,2.436616,0.001757,15.564775,0.000090,3.878750,...,-8.926347,0.019815,-6.733740,-11.859784,-8.966100,0.030090,0.001308,0.018761,-0.114062,-0.119853
50%,6.282327,14.000000,-46.745503,46720.962777,208.762655,3.594639,0.454968,22.902321,0.021097,4.317500,...,6.008148,0.368596,9.437537,3.057906,5.469837,2.159130,0.776523,0.368490,0.000192,0.000326
75%,12.917979,20.000000,-20.417815,47688.426537,212.038860,13.607880,6.972979,35.325016,0.277072,4.675625,...,12.811204,3.516586,13.174231,12.490219,12.640307,8.307202,7.744503,3.516509,0.120147,0.114204
max,20.762808,30.000000,19.105962,47970.669147,215.611941,24.033787,400.009062,41.541219,25.699636,4.962500,...,20.728720,165.593847,20.762808,20.705299,20.707836,105.303579,271.469770,165.593742,219.736612,239.286692


In [80]:
X = data.drop(columns=["rain_event","timestamp","simulation_id"])
Y = data["rain_event"]

In [81]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)

Categorical columns: ['station', 'climate']
Numerical columns: ['received_snr_db', 'carrier_frequency_ghz', 'elevation_angle_deg', 'slant_range_km', 'fspl_db', 'gaseous_attenuation_db', 'excess_attenuation_db', 'effective_path_length_km', 'specific_attenuation_db_per_km', 'rain_height_km', 'frequency_ghz', 'itu_k', 'itu_alpha', 'rain_rate_mm_per_hr', 'season_sin', 'season_cos', 'gs_latitude', 'gs_humidity', 'gs_wv', 'itu_R001', 'itu_P_rain', 'snr_roll_mean_5min', 'snr_roll_std_5min', 'snr_roll_max_5min', 'snr_roll_min_5min', 'snr_roll_mean_30min', 'snr_roll_std_30min', 'attenuation_roll_mean', 'attenuation_roll_std', 'attenuation_delta', 'snr_delta']


In [82]:
res = []
for i in data['station']:
    if i not in res:
        res.append(i)
print(res)
res = []
for i in data['climate']:
    if i not in res:
        res.append(i)
print(res)

['Delhi', 'Sao Paulo', 'Tokyo', 'Berlin']
['heavy monsoon', 'tropical', 'subtropical', 'temperate']


In [83]:
print(list(data.columns))

['timestamp', 'received_snr_db', 'carrier_frequency_ghz', 'elevation_angle_deg', 'slant_range_km', 'fspl_db', 'gaseous_attenuation_db', 'excess_attenuation_db', 'effective_path_length_km', 'specific_attenuation_db_per_km', 'rain_height_km', 'frequency_ghz', 'itu_k', 'itu_alpha', 'station', 'climate', 'simulation_id', 'rain_rate_mm_per_hr', 'rain_event', 'season_sin', 'season_cos', 'gs_latitude', 'gs_humidity', 'gs_wv', 'itu_R001', 'itu_P_rain', 'snr_roll_mean_5min', 'snr_roll_std_5min', 'snr_roll_max_5min', 'snr_roll_min_5min', 'snr_roll_mean_30min', 'snr_roll_std_30min', 'attenuation_roll_mean', 'attenuation_roll_std', 'attenuation_delta', 'snr_delta']


In [84]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=RandomState, stratify=Y)
print("Training set shape:", x_train.shape)
print("Test set shape:", x_test.shape)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

Training set shape: (72000, 33)
Test set shape: (18000, 33)
Scale pos weight: 0.86


In [85]:
categorical_cols = ["station", "climate"]
numeric_cols = [col for col in x_train.columns if col not in categorical_cols]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [86]:
models = {
    "DecisionTreeClassifier": DecisionTreeClassifier(
        splitter='random', random_state=RandomState, max_features='sqrt'
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", n_jobs=-1, random_state=RandomState
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, eval_metric="logloss",
        scale_pos_weight=scale_pos_weight, n_jobs=-1, random_state=RandomState
    ),
    "LightGBM": LGBMClassifier(
        class_weight="balanced", n_jobs=-1, random_state=RandomState, verbose=-1
    ),
}

In [87]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RandomState)

In [88]:
param_grids = {
    "DecisionTreeClassifier": {
        "classifier__max_depth": [5, 10, None],
        "classifier__criterion": ["gini", "entropy"]
    },
    "Random Forest": {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [10, None]
    },
    "XGBoost": {
        "classifier__learning_rate": [0.01, 0.1],
        "classifier__max_depth": [3, 5]
    },
    "LightGBM":{
        "classifier__n_estimators":[200, 300, 400, 500],
    }
}

In [89]:
for model_name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])
    print(sorted(pipe.get_params().keys()))

['classifier', 'classifier__ccp_alpha', 'classifier__class_weight', 'classifier__criterion', 'classifier__max_depth', 'classifier__max_features', 'classifier__max_leaf_nodes', 'classifier__min_impurity_decrease', 'classifier__min_samples_leaf', 'classifier__min_samples_split', 'classifier__min_weight_fraction_leaf', 'classifier__monotonic_cst', 'classifier__random_state', 'classifier__splitter', 'memory', 'preprocessor', 'preprocessor__cat', 'preprocessor__cat__categories', 'preprocessor__cat__drop', 'preprocessor__cat__dtype', 'preprocessor__cat__feature_name_combiner', 'preprocessor__cat__handle_unknown', 'preprocessor__cat__max_categories', 'preprocessor__cat__min_frequency', 'preprocessor__cat__sparse_output', 'preprocessor__force_int_remainder_cols', 'preprocessor__n_jobs', 'preprocessor__num', 'preprocessor__remainder', 'preprocessor__sparse_threshold', 'preprocessor__transformer_weights', 'preprocessor__transformers', 'preprocessor__verbose', 'preprocessor__verbose_feature_names

In [90]:
results = {}
for model_name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])
    start = time.time()
    scores = GridSearchCV(estimator=pipe, param_grid = param_grids[model_name], cv = cv, scoring='f1', n_jobs=-1)
    scores.fit(x_train, y_train)
    elapsed = time.time() - start
    results[model_name] = {
        "mean": scores.best_score_, "std": scores.cv_results_["std_test_score"][scores.best_index_], 
        "elapsed": elapsed, "best_params": scores.best_params_, "estimator": scores.best_estimator_}
    print(f"{model_name} - Mean Accuracy: {scores.best_score_:.4f}, Std: {scores.cv_results_['std_test_score'][scores.best_index_]}, Elapsed Time: {elapsed:.4f} seconds")
best_name = max(results, key=lambda m: results[m]["mean"])
print(f"\nBest Model: {best_name} with Mean Accuracy: {results[best_name]['mean']:.4f}, Elapsed Time: {results[best_name]['elapsed']:.2f} seconds")
print(results[best_name]["estimator"].predict(x_test))

DecisionTreeClassifier - Mean Accuracy: 0.9982, Std: 0.0006259768243899983, Elapsed Time: 10.9679 seconds
Random Forest - Mean Accuracy: 1.0000, Std: 0.0, Elapsed Time: 42.5507 seconds
XGBoost - Mean Accuracy: 0.9998, Std: 0.00012500395472075462, Elapsed Time: 11.9032 seconds
LightGBM - Mean Accuracy: 1.0000, Std: 0.0, Elapsed Time: 12.3855 seconds

Best Model: Random Forest with Mean Accuracy: 1.0000, Elapsed Time: 42.55 seconds
[1 1 1 ... 1 1 0]


In [91]:
if best_name == "XGBoost":
    tuned_model = XGBClassifier(eval_metric="logloss", scale_pos_weight=scale_pos_weight, n_jobs=-1, random_state=RandomState)
    param_dist = {
        "model__n_estimators": [200, 300, 500, 800],
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
        "model__subsample": [0.7, 0.8, 0.9, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 3, 5],
    }
else:
    tuned_model = LGBMClassifier(class_weight="balanced", n_jobs=-1, random_state=RandomState, verbose=-1)
    param_dist = {
        "model__n_estimators": [200, 300, 500, 800],
        "model__max_depth": [-1, 4, 6, 8, 10],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
        "model__num_leaves": [15, 31, 63, 127],
        "model__subsample": [0.7, 0.8, 0.9, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
    }

In [92]:
pipe = Pipeline([("preprocess", preprocessor), ("model", tuned_model)])
search = RandomizedSearchCV(
    pipe, param_distributions=param_dist, n_iter=25, cv=cv,
    scoring="roc_auc", n_jobs=-1, random_state=RandomState, verbose=1
)
search.fit(x_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


,estimator,Pipeline(step...verbose=-1))])
,param_distributions,"{'model__colsample_bytree': [0.6, 0.8, ...], 'model__learning_rate': [0.01, 0.03, ...], 'model__max_depth': [-1, 4, ...], 'model__n_estimators': [200, 300, ...], ...}"
,n_iter,25
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [93]:
print("\nBest CV AUC:", round(search.best_score_, 4))
print("Best params:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")
 
best_pipe = search.best_estimator_


Best CV AUC: 1.0
Best params:
  model__subsample: 0.7
  model__num_leaves: 127
  model__n_estimators: 300
  model__max_depth: 8
  model__learning_rate: 0.05
  model__colsample_bytree: 0.6


In [94]:
y_pred = best_pipe.predict(x_test)
y_proba = best_pipe.predict_proba(x_test)[:, 1]
 
print("\n" + "=" * 60)
print("FINAL EVALUATION ON TEST SET")
print("=" * 60)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}")
 
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
 
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Rejected (0)", "Approved (1)"]))

c:\Learning_ML\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Learning_ML\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



FINAL EVALUATION ON TEST SET
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC-AUC  : 1.0000

Confusion Matrix:
[[8302    0]
 [   0 9698]]

Classification Report:
              precision    recall  f1-score   support

Rejected (0)       1.00      1.00      1.00      8302
Approved (1)       1.00      1.00      1.00      9698

    accuracy                           1.00     18000
   macro avg       1.00      1.00      1.00     18000
weighted avg       1.00      1.00      1.00     18000



In [95]:
joblib.dump(best_pipe, "loan_model_pipeline.pkl")
print("\nSaved trained pipeline to loan_model_pipeline.pkl")


Saved trained pipeline to loan_model_pipeline.pkl
